# Image-level experiment on a Colab GPU

This notebook reproduces `results/image_level/` from `scripts/run_image_level.py`: a frozen ImageNet backbone at 150×150 with `pooling='max'`, a Dense–BatchNorm–Dropout head, keras-tuner `RandomSearch` with `max_trials=10`, `executions_per_trial=3`, `epochs=10`, then a 10-epoch refit, for VGG16, MobileNet, ResNet50, InceptionV3 and Xception. The evaluation protocol:

1. a stratified **70/15/15 train/val/test split** by image, with byte-identical files kept in one subset; hyperparameters are chosen on the validation set and the test set is used once per model;
2. each network's own **`preprocess_input`**, applied once to 0–255 pixels;
3. metrics on **argmax labels**.

**Caveat.** The split is still by image. About 97% of test images have a near-identical neighbouring slice of the same scan in training, so these scores measure performance on familiar scans, not new patients. See `results/REPORT.md` for the scan-level evaluation.

**How to use**
1. Put the repository folder on Google Drive (default `MyDrive/kidney-ct-transfer-learning`).
2. *Runtime → Change runtime type → T4 GPU*, then *Run all*.
3. After a disconnect, just *Run all* again. Everything that is slow is checkpointed to Drive (`MyDrive/kidney-image-level/`): the decoded images, the feature caches, every tuner trial and every finished model. The run continues where it stopped.

The input pipeline (decoding, resizing, `preprocess_input`) and the split and metric functions are imported from `src/kidney_tl/`, so this notebook and the local script feed the networks identically.

In [ ]:
# ---- settings ---------------------------------------------------------------
REPO_ON_DRIVE = "/content/drive/MyDrive/kidney-ct-transfer-learning"   # the repository folder
CKPT = "/content/drive/MyDrive/kidney-image-level"                     # checkpoints and results
MODE = "end_to_end"   # "end_to_end": backbone + head trained in one Keras model (frozen backbone)
                      # "cached":     run scripts/run_image_level.py (cached features, mathematically equivalent, much faster)
MODELS = ["VGG16", "MobileNet", "ResNet50", "InceptionV3", "Xception"]
SPLIT_CSV_URL = ""    # optional: raw URL of results/image_level/split.csv if it is not in the repository copy
SEED = 123

In [ ]:
# ---- mount Drive, install keras-tuner, import the shared pipeline ------------
import os, sys, json, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras-tuner", "huggingface_hub", "pyarrow"], check=True)

REPO, CKPT = Path(REPO_ON_DRIVE), Path(CKPT)
for sub in ("cache", "tuner", "results"):
    (CKPT / sub).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "scripts"))

import numpy as np
import pandas as pd
import tensorflow as tf

from kidney_tl.data import index_images, file_md5
from kidney_tl.image_level import image_level_split, assert_split_ok, evaluate
from kidney_tl.image_level_pipeline import (CLASS_INDEX, IMG_SIZE, PREPROCESSING, backbone, check_labels,
                                            decode_resize, preprocess_fn, preprocessing_stats)

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Dataset

Downloaded from the Hugging Face mirror `mhmad240/kidney-ct-classification` with the repository's own `download_data.from_huggingface`, which refuses to continue unless the class counts equal the Kaggle dataset (Cyst 3,709, Normal 5,077, Stone 1,377, Tumor 2,283). Images go to the fast local disk; the parquet download is cached on Drive.

In [ ]:
import download_data   # scripts/download_data.py

DATA = Path("/content/data/ct-kidney")
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    download_data.from_huggingface(DATA)

df = index_images(DATA)
counts = df["cls"].value_counts().to_dict()
assert counts == download_data.EXPECTED_COUNTS, counts
assert {c: i for i, c in enumerate(df.attrs["classes"])} == CLASS_INDEX
check_labels(df["path"].tolist(), df["label"].tolist())
print(len(df), "images:", counts)

## 2. Split

The same split as the local run. In order of preference: `split.csv` already on Drive, the copy in the repository (`results/image_level/split.csv`), `SPLIT_CSV_URL`, or a deterministic regeneration (stratified by class, seed 123, byte-identical files grouped by MD5). A regenerated split is checked against the saved one whenever both exist.

In [ ]:
split_path = CKPT / "split.csv"
if not split_path.exists():
    repo_copy = REPO / "results" / "image_level" / "split.csv"
    if repo_copy.exists():
        split_path.write_bytes(repo_copy.read_bytes())
    elif SPLIT_CSV_URL:
        import urllib.request
        urllib.request.urlretrieve(SPLIT_CSV_URL, split_path)

md5 = np.array([file_md5(DATA / p) for p in df["path"]])
regenerated = image_level_split(df["label"].to_numpy(), md5, seed=SEED)
if split_path.exists():
    split = pd.read_csv(split_path)
    assert split["path"].tolist() == df["path"].tolist(), "split.csv rows do not match the dataset file order"
    subset = split["subset"].to_numpy()
    print("split.csv loaded; identical to the regenerated split:", bool((subset == regenerated).all()))
else:
    subset = regenerated
    pd.DataFrame({"path": df["path"], "class": df["cls"], "subset": subset}).to_csv(split_path, index=False)
    print("split regenerated and saved to", split_path)
assert_split_ok(subset, md5)   # disjoint subsets, no byte-identical file in two subsets
print(pd.crosstab(df["cls"], subset)[["train", "val", "test"]])

## 3. Input checks

Every file is decoded to 3-channel RGB and resized bilinearly to 150×150 (aspect ratio not kept, as `image_dataset_from_directory` did). Pixels stay in 0–255 and go through the model's own `preprocess_input` exactly once: "caffe" mode (RGB→BGR, ImageNet mean subtraction) for VGG16 and ResNet50, and [-1, 1] scaling for MobileNet, InceptionV3 and Xception.

In [ ]:
files = [str(DATA / p) for p in df["path"]]
y_all = df["label"].to_numpy()
sample = tf.stack([decode_resize(files[i]) for i in np.flatnonzero(subset == "train")[:16]]).numpy()
assert sample.max() > 1.0 and sample.min() >= 0 and sample.max() <= 255, "pixels must be 0-255"
for name in MODELS:
    st = preprocessing_stats(name, sample)   # asserts the expected range
    print(f"{name:11s} {st['mode']:52s} min {st['min']:8.2f} max {st['max']:8.2f} mean {st['mean']:8.2f} std {st['std']:7.2f}")

## 4a. `MODE = "cached"`: run the local script

Runs `scripts/run_image_level.py` with its caches, tuner and results on Drive. It resumes on its own after a disconnect.

In [ ]:
if MODE == "cached":
    out = CKPT / "results"
    out.mkdir(parents=True, exist_ok=True)
    (out / "split.csv").write_bytes(split_path.read_bytes())
    cmd = [sys.executable, str(REPO / "scripts" / "run_image_level.py"), "--data-dir", str(DATA),
           "--out", str(out), "--cache-dir", str(CKPT / "cache"), "--models", *MODELS]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "WARNING" not in line:
            print(line, end="")
    assert proc.wait() == 0

## 4b. `MODE = "end_to_end"`: backbone and head in one Keras model

The decoded 150×150 images are cached once to Drive as uint8 (about 840 MB) so a reconnect does not have to decode 12,446 JPEGs again. The model: frozen backbone → Flatten → Dense(128–1024) → BatchNorm → Dropout(0.2–0.5) → Dense(4, softmax), Adam with lr in {1e-2, 1e-3, 1e-4}. Only training batches are shuffled (fixed seed).

In [ ]:
def cached_images():
    """All images decoded and resized once, as uint8, checkpointed to Drive in chunks."""
    final = CKPT / "cache" / f"images_{IMG_SIZE}_uint8.npy"
    if final.exists():
        return np.load(final, mmap_mode="r")
    part = CKPT / "cache" / f"images_{IMG_SIZE}_uint8.partial.npy"
    mark = CKPT / "cache" / f"images_{IMG_SIZE}_uint8.done.json"
    done = json.loads(mark.read_text())["done"] if (part.exists() and mark.exists()) else 0
    arr = (np.load(part, mmap_mode="r+") if done else
           np.lib.format.open_memmap(part, mode="w+", dtype=np.uint8, shape=(len(files), IMG_SIZE, IMG_SIZE, 3)))
    ds = tf.data.Dataset.from_tensor_slices(files[done:]).map(decode_resize, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    for x in ds:
        # bilinear output is fractional; round to the nearest integer for storage (max error 0.5 grey level)
        arr[done:done + len(x)] = np.clip(np.rint(x.numpy()), 0, 255).astype(np.uint8)
        done += len(x)
        arr.flush(); mark.write_text(json.dumps({"done": done}))
        print(f"\rdecoded {done}/{len(files)}", end="")
    del arr
    (CKPT / "cache" / f"images_{IMG_SIZE}_uint8.paths.txt").write_text("\n".join(df["path"]) + "\n")
    part.rename(final)
    return np.load(final, mmap_mode="r")


def make_ds(images, labels, name, shuffle):
    pre = preprocess_fn(name)
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:   # training batches only
        ds = ds.shuffle(len(labels), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(32).map(lambda x, y: (pre(tf.cast(x, tf.float32)), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)


def build_for(name):
    def build(hp):
        base = backbone(name)                      # include_top=False, 150x150x3, pooling='max', ImageNet, frozen
        model = tf.keras.Sequential([
            base,
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(hp.Int("dense_units", 128, 1024, step=128), activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(hp.Float("dropout_rate", 0.2, 0.5, step=0.1)),
            tf.keras.layers.Dense(4, activation="softmax"),
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model
    return build

In [ ]:
import keras_tuner as kt
import matplotlib.pyplot as plt

if MODE == "end_to_end":
    images = cached_images()
    rows = (CKPT / "cache" / f"images_{IMG_SIZE}_uint8.paths.txt").read_text().splitlines()
    assert rows == df["path"].tolist(), "image cache rows are not aligned with split.csv"
    idx = {s: np.flatnonzero(subset == s) for s in ("train", "val", "test")}
    data = {s: (np.asarray(images[i]), y_all[i]) for s, i in idx.items()}   # ~840 MB in RAM in total

    for name in MODELS:
        out = CKPT / "results" / name
        if (out / "metrics.json").exists():
            print(name, "already done:", json.loads((out / "metrics.json").read_text())["accuracy"]); continue
        out.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
        train_ds, val_ds = make_ds(*data["train"], name, True), make_ds(*data["val"], name, False)
        tuner = kt.RandomSearch(build_for(name), objective="val_accuracy", max_trials=10, executions_per_trial=3,
                                directory=str(CKPT / "tuner"), project_name=f"{name}_hyperparameter_tuning",
                                overwrite=False, seed=SEED)          # resumes trial by trial from Drive
        tuner.search(train_ds, validation_data=val_ds, epochs=10, verbose=1)
        best = tuner.get_best_hyperparameters(1)[0]
        print(name, "best:", best.values)

        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
        model = build_for(name)(best)
        hist = pd.DataFrame(model.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1).history)
        hist.to_csv(out / "history.csv")

        prob = model.predict(make_ds(*data["test"], name, False), verbose=0)   # the single test evaluation
        m = evaluate(data["test"][1], prob, df.attrs["classes"], n_boot=1000, seed=SEED)
        m.update(model=name, best_hyperparameters=best.values, minutes=round((time.time() - t0) / 60, 1))
        pred = pd.DataFrame({"path": df["path"].iloc[idx["test"]].to_numpy(), "y_true": data["test"][1],
                             "y_pred": prob.argmax(1), **{f"p_{c}": prob[:, i] for i, c in enumerate(df.attrs["classes"])}})
        pred.to_csv(out / "predictions.csv", index=False)
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        for a, k in zip(ax, ("loss", "accuracy")):
            a.plot(hist[k], label="training"); a.plot(hist[f"val_{k}"], label="validation"); a.set_title(k); a.legend()
        fig.savefig(out / "curves.png", dpi=110); plt.close(fig)
        (out / "metrics.json").write_text(json.dumps(m, indent=2, default=float))   # written last = model complete
        print(f"{name}: test accuracy {m['accuracy']:.4f} [{m['accuracy_ci_low']:.4f}, {m['accuracy_ci_high']:.4f}], macro-F1 {m['macro_f1']:.4f}")

## 5. Results

In [ ]:
rows = []
for name in MODELS:
    p = CKPT / "results" / name / "metrics.json"
    if p.exists():
        m = json.loads(p.read_text())
        rows.append({"model": name, "test accuracy": m["accuracy"],
                     "95% CI": f"[{m['accuracy_ci_low']:.4f}, {m['accuracy_ci_high']:.4f}]",
                     "balanced acc.": m["balanced_accuracy"], "macro-F1": m["macro_f1"],
                     "ROC-AUC": m.get("roc_auc_macro_ovr"), **{f"{c} F1": m[f"{c}_f1"] for c in df.attrs["classes"]},
                     **{f"hp {k}": v for k, v in m["best_hyperparameters"].items()}})
summary = pd.DataFrame(rows)
summary.to_csv(CKPT / "results" / "summary.csv", index=False)
summary.round(4)